<a href="https://colab.research.google.com/github/Asif-Ahmed-Rezvi/flyrank-internship-ml/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Asif-Ahmed-Rezvi/flyrank-internship-ml/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 1. Method choice and why

**Goal:** Rank content items for review based on the risk of a future impression decline, using only information available at the end of March 2026.

**Target:** `april_decline_label` — 1 when April impressions fall by more than 20% compared with March impressions; otherwise 0. This preserves the Week-3 data contract.

**First model: Logistic Regression.** This is an interpretable starting point for a binary outcome. Its predicted probabilities can rank content items, allowing evaluation with Precision@20 and Precision@50 rather than relying only on classification accuracy.

**Baseline:** Reuse the Week-4 high-visibility, low-CTR rule, calculated from March features. Evaluate its ranking alongside the model on exactly the same held-out clients. This makes the comparison fair while keeping the Week-3 target unchanged.

**Leakage safeguards:** Use only the five contracted March features: impressions, clicks, mean position, impression days, and active-day span. Do not use April metrics, `april_trend_pct`, client/content identifiers as features, or the starter dataset's current-window trend label.

**Decision rule:** Keep the learned model only if it improves useful ranking metrics enough to justify the added complexity. Otherwise, retain the transparent baseline.


In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

%pip -q install duckdb huggingface_hub scikit-learn pandas

import os
import duckdb
import numpy as np
import pandas as pd
from IPython.display import display

# Read the token from Colab Secrets or an existing environment variable.
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN is missing. In Colab, open Secrets, add a secret "
        "named HF_TOKEN, and enable notebook access."
    )

con = duckdb.connect()
con.execute(
    "CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN ?)",
    [HF_TOKEN],
)

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT_MARCH = (
    f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
)
FACT_APRIL = (
    f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')"
)

# The five features match the Week-3 contract.
FEATURES = [
    "march_impressions",
    "march_clicks",
    "march_mean_position",
    "march_impression_days",
    "march_active_days_span",
]

# Aggregate March features and April outcomes separately.
# April fields are used only to construct the target, never as features.
feature_sql = f"""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS march_impressions,
        SUM(gsc_clicks) AS march_clicks,
        AVG(NULLIF(gsc_avg_position, 0)) AS march_mean_position,
        COUNT(DISTINCT CASE
            WHEN gsc_impressions > 0 THEN report_date
        END) AS march_impression_days,
        DATE_DIFF('day', MIN(report_date), MAX(report_date)) + 1
            AS march_active_days_span
    FROM {FACT_MARCH}
    GROUP BY 1, 2
),
april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS april_impressions
    FROM {FACT_APRIL}
    GROUP BY 1, 2
)
SELECT
    m.client_hash_id,
    m.content_hash_id,
    m.march_impressions,
    m.march_clicks,
    m.march_mean_position,
    m.march_impression_days,
    m.march_active_days_span,
    a.april_impressions,
    100.0 * (a.april_impressions - m.march_impressions)
        / NULLIF(m.march_impressions, 0) AS april_trend_pct
FROM march AS m
JOIN april AS a
  ON m.client_hash_id = a.client_hash_id
 AND m.content_hash_id = a.content_hash_id
WHERE m.march_impressions > 0
"""

feature_frame = con.sql(feature_sql).df()

# Positive class: April impressions declined by more than 20%.
feature_frame["april_decline_label"] = (
    feature_frame["april_trend_pct"] < -20.0
).astype(int)

# The target and identifiers remain separate from the feature matrix.
X = feature_frame[FEATURES].copy()
y = feature_frame["april_decline_label"].astype(int)
groups = feature_frame["client_hash_id"].copy()

# Check the data contract before moving on.
print("FEATURE FRAME")
print("=============")
print(f"Content items: {len(feature_frame):,}")
print(f"Distinct clients: {groups.nunique():,}")
print(f"Decline rate: {y.mean():.1%}")
print(f"Feature columns: {X.columns.tolist()}")
print(f"Missing feature values: {int(X.isna().sum().sum()):,}")

print("\nFeature preview:")
display(feature_frame[FEATURES + ["april_decline_label"]].head())

assert X.columns.tolist() == FEATURES
assert feature_frame["content_hash_id"].is_unique
assert feature_frame["april_decline_label"].isin([0, 1]).all()

print("\nFeature and target checks passed.")


# Inspect missingness by feature before deciding how to handle it.
missing_by_feature = (
    X.isna()
    .sum()
    .rename("missing_rows")
    .to_frame()
)

missing_by_feature["missing_pct"] = (
    100 * missing_by_feature["missing_rows"] / len(X)
).round(2)

print("Missing values by feature:")
display(missing_by_feature)

print(
    "\nRows with at least one missing feature:",
    f"{int(X.isna().any(axis=1).sum()):,}",
    f"({100 * X.isna().any(axis=1).mean():.2f}%)",
)

print("\nMissing average position may mean no position measurement.")
print(
    "We will handle missing values inside the training pipeline "
    "so preprocessing is fitted on training data only."
)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FEATURE FRAME
Content items: 176,737
Distinct clients: 47
Decline rate: 53.2%
Feature columns: ['march_impressions', 'march_clicks', 'march_mean_position', 'march_impression_days', 'march_active_days_span']
Missing feature values: 1,434

Feature preview:


,march_impressions,march_clicks,march_mean_position,march_impression_days,march_active_days_span,april_decline_label
0,13.0,0.0,5.250000,4,31,1
1,6.0,0.0,11.666667,3,31,0
2,13.0,0.0,30.055556,9,31,0
3,2.0,0.0,5.000000,2,31,0
4,13.0,0.0,6.740741,9,31,1



Feature and target checks passed.
Missing values by feature:


,missing_rows,missing_pct
march_impressions,0,0.00
march_clicks,0,0.00
march_mean_position,1434,0.81
march_impression_days,0,0.00
march_active_days_span,0,0.00



Rows with at least one missing feature: 1,434 (0.81%)

Missing average position may mean no position measurement.
We will handle missing values inside the training pipeline so preprocessing is fitted on training data only.



## 2. Split design

I use a client-grouped holdout split so that no client appears in both training and test sets. This tests whether the model generalizes to clients it has not seen during training.

I use `GroupShuffleSplit` with a fixed random seed. The target is `april_decline_label`, defined by the Week-3 contract: an April impression decline greater than 20% versus March.

The five March features are the only model inputs. Missing `march_mean_position` values will be imputed inside a pipeline fitted on the training set only.

For a fair comparison, I will evaluate both Logistic Regression and the Week-4 rule on the exact same held-out rows. The main ranking metrics will be Precision@20 and Precision@50, with the test-set decline rate as the base-rate reference.

A limitation is that this is a grouped client holdout, not a later calendar-period test. It measures generalization to unseen clients in the March-to-April task, not performance on a future month.


In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.model_selection import GroupShuffleSplit

# Keep the feature frame, target, and client grouping aligned.
model_df = feature_frame.reset_index(drop=True).copy()
X = model_df[FEATURES].copy()
y = model_df["april_decline_label"].astype(int)
groups = model_df["client_hash_id"]

# Split by client, not by individual content item.
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42,
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()
y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()
groups_train = groups.iloc[train_idx].copy()
groups_test = groups.iloc[test_idx].copy()

# Verify that the same client cannot appear in both sets.
train_clients = set(groups_train.unique())
test_clients = set(groups_test.unique())
overlap_clients = train_clients & test_clients

print("GROUPED HOLDOUT SPLIT")
print("=====================")
print(f"Training rows: {len(X_train):,}")
print(f"Test rows: {len(X_test):,}")
print(f"Training clients: {len(train_clients):,}")
print(f"Test clients: {len(test_clients):,}")
print(f"Client overlap: {len(overlap_clients)}")
print(f"Training decline rate: {y_train.mean():.1%}")
print(f"Test decline rate: {y_test.mean():.1%}")
print(f"Training missing feature values: {int(X_train.isna().sum().sum()):,}")
print(f"Test missing feature values: {int(X_test.isna().sum().sum()):,}")

assert len(overlap_clients) == 0, (
    "Leakage risk: at least one client appears in both splits."
)
assert set(train_idx).isdisjoint(set(test_idx))
assert len(X_train) + len(X_test) == len(model_df)

print("\nGrouped split checks passed.")


GROUPED HOLDOUT SPLIT
Training rows: 133,473
Test rows: 43,264
Training clients: 35
Test clients: 12
Client overlap: 0
Training decline rate: 55.7%
Test decline rate: 45.3%
Training missing feature values: 1,202
Test missing feature values: 232

Grouped split checks passed.



## 3. Train + compare against my baseline

I train Logistic Regression as the first learned model because the target is binary and the method provides a relatively interpretable starting point. Numeric features are median-imputed and standardized inside a pipeline fitted only on training data.

The Week-4 rule is recalculated using March features only: pages need at least 500 impressions and a positive mean position no worse than 20, with a CTR gap below 0.5%. Since the Week-3 contract contains March clicks and impressions but no contracted CTR field, March CTR is calculated as clicks divided by impressions, multiplied by 100.

Both methods rank the same held-out content items. I compare Precision@20 and Precision@50 with the test-set decline rate as the base-rate reference. These metrics measure how many of the highest-ranked items actually meet the April-decline label.

The comparison is specific to this client-grouped holdout. It does not establish performance on a future calendar period.


In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

# ---------------------------------------------------------
# Train Logistic Regression using training data only.
# ---------------------------------------------------------
model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42,
    )),
])

model.fit(X_train, y_train)

# Probability that a content item will have an April decline.
model_prob = model.predict_proba(X_test)[:, 1]

# ---------------------------------------------------------
# Reconstruct the Week-4 rule using March features.
# Do not use April outcomes to calculate the score.
# ---------------------------------------------------------
test_features = X_test.copy()

march_impressions = test_features["march_impressions"]
march_clicks = test_features["march_clicks"]
march_position = test_features["march_mean_position"]

march_ctr = (
    100 * march_clicks / march_impressions.replace(0, np.nan)
).fillna(0)

ctr_gap = np.clip(0.5 - march_ctr, 0, 0.5) / 0.5

visibility_gate = (
    (march_impressions >= 500)
    & (march_position > 0)
    & (march_position <= 20)
).astype(int)

baseline_score = (
    np.log1p(march_impressions)
    * ctr_gap
    * visibility_gate
)

# ---------------------------------------------------------
# Ranking metrics: same held-out rows and same target.
# ---------------------------------------------------------
def precision_at_k(y_true, scores, k):
    y_array = np.asarray(y_true)
    score_array = np.asarray(scores)

    k = min(k, len(y_array))
    top_idx = np.argsort(-score_array, kind="mergesort")[:k]

    return float(y_array[top_idx].mean())

results = []

for name, scores in [
    ("Week-4 rule baseline", baseline_score.to_numpy()),
    ("Logistic Regression", model_prob),
]:
    results.append({
        "method": name,
        "precision_at_20": precision_at_k(y_test, scores, 20),
        "precision_at_50": precision_at_k(y_test, scores, 50),
        "roc_auc": roc_auc_score(y_test, scores),
        "average_precision": average_precision_score(y_test, scores),
    })

comparison = pd.DataFrame(results)

print("TEST SET BASE RATE")
print("==================")
print(f"April-decline rate: {y_test.mean():.1%}")

print("\nMODEL VS BASELINE — SAME HELD-OUT ROWS")
print("======================================")
display(comparison.round(4))

print("\nTop 20 items by Logistic Regression score:")
model_top20 = (
    test_features.assign(
        actual_april_decline=y_test.to_numpy(),
        model_score=model_prob,
        baseline_score=baseline_score.to_numpy(),
    )
    .sort_values("model_score", ascending=False)
    .head(20)
)

display(
    model_top20[
        [
            "march_impressions",
            "march_clicks",
            "march_mean_position",
            "actual_april_decline",
            "model_score",
            "baseline_score",
        ]
    ]
)


TEST SET BASE RATE
April-decline rate: 45.3%

MODEL VS BASELINE — SAME HELD-OUT ROWS


,method,precision_at_20,precision_at_50,roc_auc,average_precision
0,Week-4 rule baseline,0.65,0.56,0.5178,0.475
1,Logistic Regression,0.65,0.54,0.5779,0.515



Top 20 items by Logistic Regression score:


,march_impressions,march_clicks,march_mean_position,actual_april_decline,model_score,baseline_score
135438,134984.0,1.0,4.545582,0,0.970197,11.795416
46730,124075.0,1.0,9.385150,1,0.959870,11.709744
91928,89332.0,4.0,7.786219,1,0.907049,11.298034
149762,83834.0,1.0,11.967474,1,0.897747,11.309560
140369,71513.0,3.0,6.395691,0,0.867423,11.083867
147149,55406.0,20.0,5.359009,1,0.755285,10.133921
149575,49157.0,19.0,8.018576,1,0.725740,9.967703
149520,41986.0,12.0,6.043156,1,0.724666,10.036620
79244,46810.0,20.0,5.615196,1,0.714128,9.834935
52053,54882.0,30.0,4.252323,1,0.712440,9.719895



## 4. Errors and interpretation

Logistic Regression improves ROC-AUC and average precision over the Week-4 rule, but does not improve Precision@20 and performs slightly worse at Precision@50. Therefore, the evidence does not justify replacing the baseline for a small review queue yet.

I inspect standardized model coefficients to understand which features are associated with a higher predicted probability of an April impression decline. These coefficients describe associations in this fitted model, not causal effects.

I also inspect false positives and false negatives at a 0.5 probability threshold. A false positive is a page predicted to decline that did not meet the April-decline definition. A false negative is a page that declined but was not predicted to decline.

These errors may arise because five March summary features cannot capture every cause of impression changes. Search demand, ranking changes, seasonality, and client-specific conditions may contribute. The available features do not establish which explanation caused an individual error.

The model remains an experiment. The transparent baseline remains a reasonable operational choice unless further validation shows that the model improves the review decisions that matter.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Section 4: Interpret coefficients and inspect concrete errors.

from IPython.display import display

# ---------------------------------------------------------
# 1. Interpret Logistic Regression coefficients.
# Standardized numeric features make coefficient magnitudes
# more comparable, although they are not causal effects.
# ---------------------------------------------------------
classifier = model.named_steps["classifier"]

coefficient_table = pd.DataFrame({
    "feature": FEATURES,
    "coefficient": classifier.coef_[0],
})

coefficient_table["absolute_coefficient"] = (
    coefficient_table["coefficient"].abs()
)

coefficient_table = coefficient_table.sort_values(
    "absolute_coefficient",
    ascending=False,
).reset_index(drop=True)

print("LOGISTIC REGRESSION COEFFICIENTS")
print("================================")
display(coefficient_table.round(4))

print(
    "\nPositive coefficients are associated with a higher predicted "
    "decline probability; negative coefficients with a lower probability, "
    "conditional on the other features in this fitted model."
)

# ---------------------------------------------------------
# 2. Inspect false positives and false negatives.
# A 0.5 threshold is used for error inspection only.
# Ranking metrics above continue to use the continuous scores.
# ---------------------------------------------------------
error_review = test_features.copy()
error_review["actual"] = y_test.to_numpy()
error_review["predicted_probability"] = model_prob
error_review["predicted_class"] = (
    error_review["predicted_probability"] >= 0.5
).astype(int)

error_review["error_type"] = np.select(
    [
        (error_review["predicted_class"] == 1)
        & (error_review["actual"] == 0),

        (error_review["predicted_class"] == 0)
        & (error_review["actual"] == 1),
    ],
    ["false_positive", "false_negative"],
    default="correct",
)

print("\nERROR COUNTS AT PROBABILITY THRESHOLD 0.5")
print("=========================================")
display(
    error_review["error_type"]
    .value_counts()
    .rename_axis("error_type")
    .to_frame("count")
)

error_columns = FEATURES + [
    "actual",
    "predicted_probability",
    "error_type",
]

false_positives = (
    error_review[error_review["error_type"] == "false_positive"]
    .sort_values("predicted_probability", ascending=False)
    .head(3)
)

false_negatives = (
    error_review[error_review["error_type"] == "false_negative"]
    .sort_values("predicted_probability", ascending=False)
    .head(3)
)

print("\nTHREE HIGH-CONFIDENCE FALSE POSITIVES")
display(false_positives[error_columns].round(4))

print("\nTHREE FALSE NEGATIVES CLOSEST TO THE DECISION THRESHOLD")
display(false_negatives[error_columns].round(4))

print(
    "\nInterpretation reminder: these are examples of prediction errors, "
    "not evidence that a particular feature caused an April decline."
)


LOGISTIC REGRESSION COEFFICIENTS


,feature,coefficient,absolute_coefficient
0,march_clicks,-0.5654,0.5654
1,march_active_days_span,0.3750,0.3750
2,march_mean_position,-0.1588,0.1588
3,march_impressions,0.1293,0.1293
4,march_impression_days,-0.0372,0.0372



Positive coefficients are associated with a higher predicted decline probability; negative coefficients with a lower probability, conditional on the other features in this fitted model.

ERROR COUNTS AT PROBABILITY THRESHOLD 0.5


,count
error_type,
correct,22713
false_positive,15505
false_negative,5046



THREE HIGH-CONFIDENCE FALSE POSITIVES


,march_impressions,march_clicks,march_mean_position,march_impression_days,march_active_days_span,actual,predicted_probability,error_type
135438,134984.0,1.0,4.5456,31,31,0,0.9702,false_positive
140369,71513.0,3.0,6.3957,31,31,0,0.8674,false_positive
137992,28950.0,0.0,9.4199,31,31,0,0.7070,false_positive



THREE FALSE NEGATIVES CLOSEST TO THE DECISION THRESHOLD


,march_impressions,march_clicks,march_mean_position,march_impression_days,march_active_days_span,actual,predicted_probability,error_type
3755,226.0,0.0,30.0120,31,31,1,0.5,false_negative
86463,630.0,1.0,5.8772,17,29,1,0.5,false_negative
36473,46.0,0.0,7.0837,16,29,1,0.5,false_negative



Interpretation reminder: these are examples of prediction errors, not evidence that a particular feature caused an April decline.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.